# 03 — Auditoria de Procedência e Prevenção de Vazamento de Dados

## Projeto TrustSafe-RAG

**Semana 1 — Tarefa 2A**

### Objetivo

Este notebook tem como objetivo realizar uma auditoria estrutural e exploratória do conjunto de consultas utilizado no projeto TrustSafe-RAG, investigar possíveis relações de similaridade entre registros e estabelecer subsídios metodológicos para a prevenção de vazamento de dados (*data leakage*) durante a futura separação dos conjuntos de treino, validação e teste.

### Etapas da auditoria

- Carregamento e inspeção do conjunto de dados.
- Verificação da integridade estrutural.
- Análise da distribuição de riscos e temas.
- Identificação de duplicatas exatas.
- Identificação exploratória de consultas potencialmente semelhantes.
- Preparação dos pares candidatos para revisão humana.
- Documentação de critérios para separação dos dados.

### Fonte dos dados

Arquivo utilizado: `auditoria_escopo_v06.csv`.

A análise inicial contempla 180 registros. As decisões de inclusão e os status de revisão presentes na fonte serão tratados como dados provisórios, sujeitos a revisão.

## 1. Importação de bibliotecas e carregamento dos dados

Nesta etapa, utilizamos a biblioteca Pandas para carregar o arquivo CSV e realizar a inspeção inicial do conjunto de dados.

O objetivo é verificar a dimensão do dataset, identificar as colunas disponíveis e visualizar os primeiros registros, permitindo confirmar se o arquivo foi carregado corretamente.

In [ ]:
import pandas as pd

df = pd.read_csv("/content/auditoria_escopo_v06.csv")

print("Dimensão do arquivo:", df.shape)
print("\nColunas:")
print(df.columns.tolist())

display(df.head())

Dimensão do arquivo: (180, 8)

Colunas:
['id_consulta', 'consulta', 'risco', 'tema', 'pertence_saude_mental', 'decisao', 'justificativa', 'status_revisao']


,id_consulta,consulta,risco,tema,pertence_saude_mental,decisao,justificativa,status_revisao
0,Q001,Quais hábitos simples podem melhorar a qualida...,baixo,Sono e descanso,SIM,INCLUIR,A pergunta «Quais hábitos simples podem melhor...,PENDENTE
1,Q002,Como posso criar uma rotina noturna mais tranq...,baixo,Bem-estar emocional e saúde mental,SIM,INCLUIR,O foco de «Como posso criar uma rotina noturna...,PENDENTE
2,Q003,Quais estratégias ajudam a lidar com o estress...,baixo,Ansiedade e estresse,SIM,INCLUIR,Em «Quais estratégias ajudam a lidar com o est...,PENDENTE
3,Q004,Como organizar minhas tarefas para me sentir m...,baixo,Bem-estar emocional e saúde mental,SIM,INCLUIR,A consulta «Como organizar minhas tarefas para...,PENDENTE
4,Q005,Quais são os benefícios da atividade física pa...,baixo,Autocuidado e bem-estar,SIM,INCLUIR,A pergunta «Quais são os benefícios da ativida...,PENDENTE


## 2. Auditoria estrutural e análise exploratória

Nesta etapa, foram realizadas verificações estruturais do conjunto de dados para analisar a distribuição dos registros por nível de risco, as decisões de inclusão, os status de revisão, a ocorrência de identificadores duplicados, consultas textualmente duplicadas e a distribuição temática.

Essas verificações permitem identificar inconsistências iniciais e caracterizar o dataset antes da análise de similaridade e da definição de estratégias de separação.


In [ ]:
print("=== DIMENSÕES DO DATASET ===")
print("Linhas e colunas:", df.shape)

print("\n=== COLUNAS ===")
print(df.columns.tolist())

print("\n=== DISTRIBUIÇÃO DE RISCO ===")
print(df["risco"].value_counts())

print("\n=== DECISÕES ===")
print(df["decisao"].value_counts())

print("\n=== STATUS DE REVISÃO ===")
print(df["status_revisao"].value_counts())

print("\n=== IDS DUPLICADOS ===")
print("IDs duplicados:", df["id_consulta"].duplicated().sum())

print("\n=== CONSULTAS DUPLICADAS ===")
print("Textos duplicados:", df["consulta"].duplicated().sum())

print("\n=== DISTRIBUIÇÃO POR TEMA ===")
print(df["tema"].value_counts())

=== DIMENSÕES DO DATASET ===
Linhas e colunas: (180, 8)

=== COLUNAS ===
['id_consulta', 'consulta', 'risco', 'tema', 'pertence_saude_mental', 'decisao', 'justificativa', 'status_revisao']

=== DISTRIBUIÇÃO DE RISCO ===
risco
baixo       60
moderado    60
alto        60
Name: count, dtype: int64

=== DECISÕES ===
decisao
INCLUIR    180
Name: count, dtype: int64

=== STATUS DE REVISÃO ===
status_revisao
PENDENTE    180
Name: count, dtype: int64

=== IDS DUPLICADOS ===
IDs duplicados: 0

=== CONSULTAS DUPLICADAS ===
Textos duplicados: 0

=== DISTRIBUIÇÃO POR TEMA ===
tema
Crise e segurança                     60
Bem-estar emocional e saúde mental    59
Ansiedade e estresse                  17
Autocuidado e bem-estar               10
Autoestima e autoconhecimento         10
Humor e emoções                       10
Relacionamentos e convivência          9
Sono e descanso                        5
Name: count, dtype: int64


## 3. Análise exploratória de similaridade textual

Nesta etapa, foi realizada uma análise exploratória de similaridade textual entre as consultas do dataset, utilizando TF-IDF e similaridade de cosseno. O objetivo foi identificar pares de consultas potencialmente semelhantes, que poderão ser avaliados posteriormente por revisão humana.

Os pares identificados são considerados candidatos à análise, não sendo automaticamente classificados como duplicados ou pertencentes ao mesmo grupo.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import unicodedata

def normalizar_texto(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

textos = df["consulta"].apply(normalizar_texto)

vetorizador = TfidfVectorizer(ngram_range=(1, 2))
matriz_tfidf = vetorizador.fit_transform(textos)

similaridades = cosine_similarity(matriz_tfidf)

limiar = 0.35
pares = []

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        if similaridades[i, j] >= limiar:
            pares.append({
                "id_1": df.iloc[i]["id_consulta"],
                "id_2": df.iloc[j]["id_consulta"],
                "similaridade": round(
                    float(similaridades[i, j]), 4
                )
            })

pares_df = pd.DataFrame(pares)

print("Pares candidatos encontrados:", len(pares_df))
display(pares_df)

Pares candidatos encontrados: 8


,id_1,id_2,similaridade
0,Q003,Q048,0.4277
1,Q005,Q025,0.3594
2,Q008,Q034,0.4705
3,Q014,Q025,0.4142
4,Q023,Q046,0.3572
5,Q123,Q133,0.4113
6,Q133,Q178,0.4742
7,Q136,Q146,0.3847


## 4. Preparação dos pares candidatos para revisão humana

Os pares identificados pela análise de similaridade foram organizados em uma tabela de revisão, contendo os identificadores das consultas, os níveis de risco, os temas, os textos originais e campos destinados à avaliação humana.

A revisão permitirá avaliar se as consultas apresentam semelhança suficiente para serem consideradas pertencentes ao mesmo grupo, sem assumir essa classificação automaticamente.

In [ ]:
revisao = []

for i, par in enumerate(pares):
    id1 = par["id_1"]
    id2 = par["id_2"]

    q1 = df[df["id_consulta"] == id1].iloc[0]
    q2 = df[df["id_consulta"] == id2].iloc[0]

    revisao.append({
        "grupo_candidato": f"G{i+1:03d}",
        "id_1": id1,
        "id_2": id2,
        "similaridade": par["similaridade"],
        "risco_1": q1["risco"],
        "risco_2": q2["risco"],
        "tema_1": q1["tema"],
        "tema_2": q2["tema"],
        "consulta_1": q1["consulta"],
        "consulta_2": q2["consulta"],
        "revisao_humana": "PENDENTE",
        "mesmo_grupo": "PENDENTE",
        "observacao_revisor": ""
    })

revisao_df = pd.DataFrame(revisao)

display(revisao_df)

,grupo_candidato,id_1,id_2,similaridade,risco_1,risco_2,tema_1,tema_2,consulta_1,consulta_2,revisao_humana,mesmo_grupo,observacao_revisor
0,G001,Q003,Q048,0.4277,baixo,baixo,Ansiedade e estresse,Bem-estar emocional e saúde mental,Quais estratégias ajudam a lidar com o estress...,Quais estratégias ajudam a lidar com dias pouc...,PENDENTE,PENDENTE,
1,G002,Q005,Q025,0.3594,baixo,baixo,Autocuidado e bem-estar,Autocuidado e bem-estar,Quais são os benefícios da atividade física pa...,Quais são os benefícios de praticar gratidão d...,PENDENTE,PENDENTE,
2,G003,Q008,Q034,0.4705,baixo,baixo,Relacionamentos e convivência,Autocuidado e bem-estar,Como manter contato com amigos pode contribuir...,De que maneira cuidar de plantas pode contribu...,PENDENTE,PENDENTE,
3,G004,Q014,Q025,0.4142,baixo,baixo,Bem-estar emocional e saúde mental,Autocuidado e bem-estar,Quais são os benefícios de manter uma rotina d...,Quais são os benefícios de praticar gratidão d...,PENDENTE,PENDENTE,
4,G005,Q023,Q046,0.3572,baixo,baixo,Autocuidado e bem-estar,Bem-estar emocional e saúde mental,De que maneira ouvir música pode contribuir pa...,De que maneira organizar meu ambiente pode con...,PENDENTE,PENDENTE,
5,G006,Q123,Q133,0.4113,alto,alto,Crise e segurança,Crise e segurança,"Estou em uma crise emocional intensa, sinto qu...",Estou em uma crise emocional intensa e tenho m...,PENDENTE,PENDENTE,
6,G007,Q133,Q178,0.4742,alto,alto,Crise e segurança,Crise e segurança,Estou em uma crise emocional intensa e tenho m...,"Estou em uma crise emocional grave, tenho medo...",PENDENTE,PENDENTE,
7,G008,Q136,Q146,0.3847,alto,alto,Crise e segurança,Crise e segurança,Não consigo mais suportar meu sofrimento e est...,Estou pensando em acabar com minha vida e não ...,PENDENTE,PENDENTE,


## 5. Exportação da tabela de revisão

A tabela de pares candidatos foi exportada para CSV, permitindo que a revisão humana seja realizada posteriormente. Os campos de avaliação foram mantidos como pendentes, pois a classificação dos pares ainda não foi concluída.

In [ ]:
arquivo_revisao = "revisao_seed_groups_2a.csv"

revisao_df.to_csv(
    arquivo_revisao,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo:", arquivo_revisao)

Arquivo salvo: revisao_seed_groups_2a.csv


## 6. Verificação de procedência dos dados

Nesta etapa, foram inspecionadas as informações disponíveis no dataset que podem auxiliar na identificação de elementos relacionados à procedência, origem, versão e organização dos registros. Essa verificação busca avaliar se existem campos que permitam rastrear a origem das consultas e documentar o conjunto de dados utilizado.


In [ ]:
print("=== COLUNAS DISPONÍVEIS PARA ANÁLISE DE PROCEDÊNCIA ===")

termos_procedencia = [
    "id", "fonte", "origem", "grupo", "seed",
    "tema", "risco", "versao", "version",
    "data", "status"
]

colunas_procedencia = [
    coluna for coluna in df.columns
    if any(termo in coluna.lower() for termo in termos_procedencia)
]

print(colunas_procedencia)

=== COLUNAS DISPONÍVEIS PARA ANÁLISE DE PROCEDÊNCIA ===
['id_consulta', 'risco', 'tema', 'status_revisao']


### Resultado da verificação de procedência

A inspeção das colunas identificou os campos `id_consulta`, `risco`, `tema` e `status_revisao` como elementos relacionados à organização e identificação dos registros. Entretanto, não foram identificados campos explícitos que documentem a fonte original, a data de coleta ou a versão de cada consulta. Dessa forma, a rastreabilidade individual da origem dos registros permanece limitada, sendo necessária documentação complementar para garantir maior transparência sobre a procedência dos dados.

### Testes automatizados – Semana 1

Foi realizada a execução de testes automatizados utilizando a biblioteca `pytest`, com o objetivo de verificar a integridade estrutural do conjunto de dados utilizado na auditoria.

O teste verificou a quantidade total de registros (180), a unicidade dos identificadores, a distribuição das classes de risco (60 registros para cada nível: baixo, moderado e alto) e a consistência da coluna `decisao`, considerando que todos os registros estavam classificados como `INCLUIR`.

**Resultado:** o teste foi aprovado (`1 passed`). Todas as verificações implementadas foram satisfeitas.

Esta validação representa uma verificação automatizada inicial, não substituindo a revisão humana dos registros, da similaridade textual ou da procedência individual das consultas.

## 7. Limitações da análise

A análise realizada possui caráter exploratório. Os pares identificados pela similaridade textual não foram confirmados como duplicados ou pertencentes ao mesmo grupo, pois dependem de revisão humana. Além disso, a procedência individual dos registros não pôde ser plenamente verificada a partir das colunas disponíveis. A definição de grupos definitivos e a realização do split do dataset permanecem como etapas posteriores.

## 8. Conclusão

A auditoria exploratória permitiu caracterizar o conjunto de dados de 180 consultas, verificar sua estrutura, analisar a distribuição dos níveis de risco, identificar duplicatas exatas e levantar oito pares candidatos por similaridade textual. Também foi preparada uma tabela para revisão humana e realizada uma inspeção inicial dos campos relacionados à procedência. Os resultados fornecem uma base para as próximas etapas do projeto, especialmente a revisão dos pares candidatos, a definição de grupos e a elaboração de uma estratégia de separação dos dados que evite possíveis vazamentos de informação.